# StockLens Pro Model Notebook
Simple function-based workflow for stock direction prediction with Random Forest.

In [ ]:
import pandas as pd
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier

SUPPORTED_TICKERS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOG', '^GSPC']
FEATURE_COLUMNS = ['prev_day_return', 'ma_5', 'ma_10', 'rolling_volatility']

In [ ]:
def validate_ticker(ticker):
    normalized = ticker.strip().upper()
    if normalized not in SUPPORTED_TICKERS:
        raise ValueError(f'Unsupported ticker {ticker}. Choose from {SUPPORTED_TICKERS}')
    return normalized


def fetch_stock_data(ticker, period='15y', interval='1d'):
    normalized = validate_ticker(ticker)
    df = yf.download(normalized, period=period, interval=interval, auto_adjust=True, progress=False)

    if df.empty:
        raise ValueError(f'No data returned for {normalized}')

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df.columns = [str(col).lower() for col in df.columns]
    df['ticker'] = normalized
    return df

In [ ]:
def build_features(df):
    data = df.copy()

    daily_return = data['close'].pct_change()
    data['prev_day_return'] = daily_return.shift(1)
    data['ma_5'] = data['close'].shift(1).rolling(5).mean()
    data['ma_10'] = data['close'].shift(1).rolling(10).mean()
    data['rolling_volatility'] = daily_return.shift(1).rolling(10).std()

    data['next_day_return'] = data['close'].shift(-1) / data['close'] - 1
    data['target'] = (data['next_day_return'] > 0).astype(int)

    data = data.dropna().reset_index(drop=True)
    return data

In [ ]:
def split_time_series(df, train_ratio=0.8):
    split_index = int(len(df) * train_ratio)
    train_df = df.iloc[:split_index].copy()
    test_df = df.iloc[split_index:].copy()
    return train_df, test_df


def get_model():
    return RandomForestClassifier(n_estimators=120, random_state=42, min_samples_leaf=2)

In [ ]:
def train_model_for_ticker(ticker):
    raw_df = fetch_stock_data(ticker)
    feature_df = build_features(raw_df)
    train_df, test_df = split_time_series(feature_df, train_ratio=0.8)

    model = get_model()
    x_train = train_df[FEATURE_COLUMNS]
    y_train = train_df['target']
    model.fit(x_train, y_train)

    x_test = test_df[FEATURE_COLUMNS]
    y_test = test_df['target']
    test_predictions = model.predict(x_test)
    accuracy = (test_predictions == y_test).mean()

    summary = {
        'ticker': validate_ticker(ticker),
        'train_rows': len(train_df),
        'test_rows': len(test_df),
        'test_accuracy': round(float(accuracy), 4),
    }

    return model, feature_df, summary

In [ ]:
def predict_latest(model, feature_df, ticker):
    latest_row = feature_df.iloc[-1]
    x_latest = latest_row[FEATURE_COLUMNS].to_frame().T

    prediction_num = int(model.predict(x_latest)[0])
    probabilities = model.predict_proba(x_latest)[0]

    return {
        'ticker': validate_ticker(ticker),
        'prediction': 'UP' if prediction_num == 1 else 'DOWN',
        'confidence': round(float(max(probabilities)), 4),
        'latest_close': round(float(latest_row['close']), 2),
        'latest_date': str(latest_row['date']).split(' ')[0],
        'volatility_score': round(float(latest_row['rolling_volatility']) * 100, 2),
    }

In [ ]:
def run_backtest(model, feature_df):
    _, test_df = split_time_series(feature_df, train_ratio=0.8)

    x_test = test_df[FEATURE_COLUMNS]
    predictions = model.predict(x_test).tolist()

    actual = test_df['target'].tolist()
    total = len(actual)
    correct = sum(1 for i in range(total) if predictions[i] == actual[i])
    accuracy = correct / total

    market_returns = test_df['next_day_return'].tolist()
    strategy_returns = [market_returns[i] if predictions[i] == 1 else 0.0 for i in range(total)]

    cumulative_market = 1.0
    cumulative_strategy = 1.0

    for market_return in market_returns:
        cumulative_market *= 1 + market_return

    for strategy_return in strategy_returns:
        cumulative_strategy *= 1 + strategy_return

    return {
        'accuracy': round(accuracy, 4),
        'total_predictions': total,
        'correct_predictions': correct,
        'strategy_return_pct': round((cumulative_strategy - 1) * 100, 2),
        'market_return_pct': round((cumulative_market - 1) * 100, 2),
        'from_date': str(test_df.iloc[0]['date']).split(' ')[0],
        'to_date': str(test_df.iloc[-1]['date']).split(' ')[0],
    }

In [ ]:
ticker = 'AAPL'

model, feature_df, train_summary = train_model_for_ticker(ticker)
latest_prediction = predict_latest(model, feature_df, ticker)
backtest_summary = run_backtest(model, feature_df)

print('Train Summary:', train_summary)
print('Latest Prediction:', latest_prediction)
print('Backtest Summary:', backtest_summary)